# 手撕 LogitsProcessor 框架（HuggingFace 风格）

## 背景
HuggingFace 的 LogitsProcessorList 是一个可组合的 logits 处理 pipeline。
每个 processor 接收 (input_ids, scores) 返回处理后的 scores。
支持温度、top-k、top-p、repetition penalty 等任意组合。

## 考察点
- pipeline 模式的设计（可组合、可插拔）
- __call__ 接口约定
- 执行顺序的重要性

In [ ]:
import torch
import torch.nn.functional as F

class LogitsProcessor:
    def __call__(self, input_ids: torch.Tensor, scores: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

class TemperatureProcessor(LogitsProcessor):
    def __init__(self, temperature: float = 1.0) -> None:
        self.t = temperature
    def __call__(self, input_ids: torch.Tensor, scores: torch.Tensor) -> torch.Tensor:
        return scores / self.t

class TopKProcessor(LogitsProcessor):
    def __init__(self, k: int = 50) -> None:
        self.k = k
    def __call__(self, input_ids: torch.Tensor, scores: torch.Tensor) -> torch.Tensor:
        values, _ = torch.topk(scores, self.k)
        scores[scores < values[-1]] = float('-inf')
        return scores

class TopPProcessor(LogitsProcessor):
    def __init__(self, p: float = 0.9) -> None:
        self.p = p
    def __call__(self, input_ids: torch.Tensor, scores: torch.Tensor) -> torch.Tensor:
        sorted_scores, sorted_idx = torch.sort(scores, descending=True)
        cum_probs = F.softmax(sorted_scores, dim=-1).cumsum(dim=-1)
        mask = cum_probs > self.p
        mask[0] = False  # 至少保留 1 个
        sorted_scores[mask] = float('-inf')
        return sorted_scores.scatter(0, sorted_idx, sorted_scores)

class RepetitionPenaltyProcessor(LogitsProcessor):
    def __init__(self, penalty: float = 1.2) -> None:
        self.penalty = penalty
    def __call__(self, input_ids: torch.Tensor, scores: torch.Tensor) -> torch.Tensor:
        for tid in set(input_ids.tolist()):
            if scores[tid] > 0: scores[tid] /= self.penalty
            else: scores[tid] *= self.penalty
        return scores

class LogitsProcessorList:
    def __init__(self, processors: list = None) -> None:
        self.processors = processors or []
    def append(self, processor: nn.Module) -> None:
        self.processors.append(processor)
    def __call__(self, input_ids: torch.Tensor, scores: torch.Tensor) -> torch.Tensor:
        for p in self.processors:
            scores = p(input_ids, scores)
        return scores

In [ ]:
# 验证组合 pipeline
torch.manual_seed(42)
vocab_size = 100
logits = torch.randn(vocab_size)
input_ids = torch.tensor([0, 2, 5, 2, 8])
# 构建 pipeline: temperature → top-k → top-p → repetition penalty
pipeline = LogitsProcessorList([
    TemperatureProcessor(temperature=0.8),
    TopKProcessor(k=20),
    TopPProcessor(p=0.9),
    RepetitionPenaltyProcessor(penalty=1.5),
])
processed = pipeline(input_ids, logits.clone())
# 应有部分 token 被 mask
n_valid = (processed > float('-inf')).sum().item()
assert n_valid <= 20, "top-k 后最多 20 个有效"
# 验证 repetition penalty 生效（token 2 应被降权）
print(f"原始 logits[2]: {logits[2]:.4f}")
print(f"处理后有效 token 数: {n_valid}")
# 单独验证每个 processor
temp_out = TemperatureProcessor(0.8)(input_ids, logits.clone())
assert torch.allclose(temp_out, logits / 0.8), "温度应除法"
print("✅ LogitsProcessor pipeline 验证通过")